============================================================
NOTEBOOK 02 — GEOSPATIAL CHOROPLETH MAP
India Road Accident Risk Map using Folium + GeoPandas
============================================================


In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from folium import Choropleth, GeoJson
import json
import requests
import warnings
warnings.filterwarnings('ignore')

CLEAN = '../data/cleaned'
OUTPUT = '../outputs'

import os
os.makedirs(OUTPUT, exist_ok=True)

print("All imports OK")


All imports OK


In [2]:
state_master = pd.read_csv(f'{CLEAN}/state_master.csv')
latest = state_master[state_master['year'] == 2024].copy()
latest = latest.dropna(subset=['accidents', 'fatalities'])
print(f"States in 2024 data: {latest['state'].nunique()}")
print(latest[['state','accidents','fatalities','fatality_rate_per_accident']].head())


States in 2024 data: 36
                        state  accidents  fatalities  \
5   Andaman & Nicobar Islands      135.0        29.0   
11             Andhra Pradesh    19557.0      8346.0   
17          Arunachal Pradesh      277.0       168.0   
23                      Assam     7848.0      3351.0   
29                      Bihar    11610.0      9347.0   

    fatality_rate_per_accident  
5                       0.2148  
11                      0.4268  
17                      0.6065  
23                      0.4270  
29                      0.8051  


In [3]:
GEOJSON_URL = 'https://raw.githubusercontent.com/datta07/INDIAN-SHAPEFILES/master/INDIA/INDIA_STATES.geojson'
r = requests.get(GEOJSON_URL, timeout=30)
india_geo = r.json()

# Save locally so you don't re-download every run
with open(f'{OUTPUT}/india_states.geojson', 'w') as f:
    json.dump(india_geo, f)

print(f"GeoJSON loaded: {len(india_geo['features'])} states/UTs")


GeoJSON loaded: 37 states/UTs


In [4]:
# GeoJSON uses 'STNAME_SH' column — check exact names first:
geo_names = sorted([f['properties']['STNAME_SH'] for f in india_geo['features']])
print("GeoJSON names:", geo_names)


GeoJSON names: ['Andaman & Nicobar', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra & Nagar Haveli', 'Daman & Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [5]:
# Some GeoJSON names differ from your cleaned data — this mapping fixes all mismatches
GEO_TO_MASTER = {
    'Andaman & Nicobar':       'Andaman & Nicobar Islands',
    'Andhra Pradesh':          'Andhra Pradesh',
    'Arunachal Pradesh':       'Arunachal Pradesh',
    'Assam':                   'Assam',
    'Bihar':                   'Bihar',
    'Chandigarh':              'Chandigarh',
    'Chhattisgarh':            'Chhattisgarh',
    'Dadra & Nagar Haveli':    'D&N Haveli and Daman & Diu',
    'Daman & Diu':             'D&N Haveli and Daman & Diu',  # merged UT
    'Delhi':                   'Delhi',
    'Goa':                     'Goa',
    'Gujarat':                 'Gujarat',
    'Haryana':                 'Haryana',
    'Himachal Pradesh':        'Himachal Pradesh',
    'Jammu & Kashmir':         'Jammu and Kashmir',
    'Jharkhand':               'Jharkhand',
    'Karnataka':               'Karnataka',
    'Kerala':                  'Kerala',
    'Ladakh':                  'Ladakh',
    'Lakshadweep':             'Lakshadweep',
    'Madhya Pradesh':          'Madhya Pradesh',
    'Maharashtra':             'Maharashtra',
    'Manipur':                 'Manipur',
    'Meghalaya':               'Meghalaya',
    'Mizoram':                 'Mizoram',
    'Nagaland':                'Nagaland',
    'Odisha':                  'Odisha',
    'Puducherry':              'Puducherry',
    'Punjab':                  'Punjab',
    'Rajasthan':               'Rajasthan',
    'Sikkim':                  'Sikkim',
    'Tamil Nadu':              'Tamil Nadu',
    'Telangana':               'Telangana',
    'Tripura':                 'Tripura',
    'Uttar Pradesh':           'Uttar Pradesh',
    'Uttarakhand':             'Uttarakhand',
    'West Bengal':             'West Bengal',
}

# Apply mapping to GeoJSON features
for feature in india_geo['features']:
    geo_name = feature['properties']['STNAME_SH']
    feature['properties']['state'] = GEO_TO_MASTER.get(geo_name, geo_name)

print("Name mapping applied.")


Name mapping applied.


In [6]:
# Build a lookup dict for fast access in tooltip
data_lookup = latest.set_index('state')[
    ['accidents','fatalities','fatality_rate_per_accident']
].to_dict('index')

print("Data lookup ready. Sample:")
print(list(data_lookup.items())[:2])


Data lookup ready. Sample:
[('Andaman & Nicobar Islands', {'accidents': 135.0, 'fatalities': 29.0, 'fatality_rate_per_accident': 0.2148}), ('Andhra Pradesh', {'accidents': 19557.0, 'fatalities': 8346.0, 'fatality_rate_per_accident': 0.4268})]


In [7]:
# Center of India: 20.5937, 78.9629
m1 = folium.Map(
    location=[20.5937, 78.9629],
    zoom_start=5,
    tiles='CartoDB positron',   # clean light background
    prefer_canvas=True
)

# Build a dataframe for choropleth (needs state column + value column)
choropleth_data = latest[['state', 'fatality_rate_per_accident', 'accidents', 'fatalities']].copy()
choropleth_data['fatality_rate_per_accident'] = choropleth_data['fatality_rate_per_accident'].round(4)

# Save merged geojson for choropleth key
with open(f'{OUTPUT}/india_mapped.geojson', 'w') as f:
    json.dump(india_geo, f)

# Add choropleth layer — colored by fatality rate per accident
Choropleth(
    geo_data=f'{OUTPUT}/india_mapped.geojson',
    data=choropleth_data,
    columns=['state', 'fatality_rate_per_accident'],
    key_on='feature.properties.state',
    fill_color='RdYlGn_r',     # Red=dangerous, Green=safer
    fill_opacity=0.8,
    line_opacity=0.3,
    legend_name='Fatality Rate per Accident (2024) — Higher = More Dangerous',
    nan_fill_color='lightgray',
    nan_fill_opacity=0.4,
    highlight=True,
).add_to(m1)


In [8]:
tooltip_style = """
    background-color: #1a1a2e;
    color: white;
    font-family: Arial, sans-serif;
    font-size: 13px;
    padding: 8px 12px;
    border-radius: 6px;
    border: 1px solid #444;
"""

def style_function(feature):
    return {
        'fillOpacity': 0,
        'weight': 0,
    }

def highlight_function(feature):
    return {
        'fillOpacity': 0.2,
        'weight': 2,
        'color': '#333',
    }

GeoJson(
    india_geo,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['state'],
        aliases=['State:'],
        localize=True,
        sticky=False,
        labels=True,
        style=tooltip_style,
        max_width=300,
    ),
    popup=folium.GeoJsonPopup(
        fields=['state'],
        aliases=['State'],
        localize=True,
    )
).add_to(m1)


In [9]:
# Top 10 by fatality rate
top10 = latest.nlargest(10, 'fatality_rate_per_accident')

# Approximate state centroids (lat, lon) for marker placement
STATE_CENTROIDS = {
    'Bihar':            (25.0961, 85.3131),
    'Jharkhand':        (23.6102, 85.2799),
    'Punjab':           (31.1471, 75.3412),
    'Uttarakhand':      (30.0668, 79.0193),
    'Uttar Pradesh':    (26.8467, 80.9462),
    'Odisha':           (20.9517, 85.0985),
    'Gujarat':          (22.2587, 71.1924),
    'West Bengal':      (22.9868, 87.8550),
    'Haryana':          (29.0588, 76.0856),
    'Rajasthan':        (27.0238, 74.2179),
    'Tamil Nadu':       (11.1271, 78.6569),
    'Karnataka':        (15.3173, 75.7139),
    'Maharashtra':      (19.7515, 75.7139),
    'Madhya Pradesh':   (22.9734, 78.6569),
    'Kerala':           (10.8505, 76.2711),
    'Andhra Pradesh':   (15.9129, 79.7400),
    'Telangana':        (18.1124, 79.0193),
}

for _, row in top10.iterrows():
    state = row['state']
    if state not in STATE_CENTROIDS:
        continue
    lat, lon = STATE_CENTROIDS[state]
    rate = row['fatality_rate_per_accident']
    accidents = int(row['accidents'])
    fatalities = int(row['fatalities'])

    # Color based on severity
    if rate >= 0.7:
        color = 'red'
    elif rate >= 0.5:
        color = 'orange'
    else:
        color = 'yellow'

    folium.CircleMarker(
        location=[lat, lon],
        radius=10,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"""
            <div style='font-family:Arial; width:200px'>
            <b style='font-size:14px'>{state}</b><br>
            <hr style='margin:4px 0'>
            <b>Fatality Rate:</b> {rate:.4f}<br>
            <b>Accidents 2024:</b> {accidents:,}<br>
            <b>Fatalities 2024:</b> {fatalities:,}<br>
            <b>Risk Rank:</b> #{list(top10['state']).index(state)+1} in India
            </div>
            """,
            max_width=220
        ),
        tooltip=f"{state}: {rate:.4f} fatality rate"
    ).add_to(m1)

# Title
title_html = '''
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
     z-index:1000; background-color:white; padding: 10px 20px;
     border-radius:8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
     font-family: Arial; text-align:center;">
    <b style="font-size:16px">India Road Accident Risk Map — 2024</b><br>
    <span style="font-size:12px; color:#666">Fatality Rate per Accident | Red markers = Top 10 highest-risk states</span>
</div>
'''
m1.get_root().html.add_child(folium.Element(title_html))

m1.save(f'{OUTPUT}/india_risk_map.html')
print("MAP 1 SAVED: outputs/india_risk_map.html")
print("Open this file in your browser — it's fully interactive.")


MAP 1 SAVED: outputs/india_risk_map.html
Open this file in your browser — it's fully interactive.


In [10]:
# ============================================================
# 02_geospatial_map.ipynb — REPLACE CELL 10 WITH THIS
# ============================================================
# Changes made:
# 1. State name labels visible on map WITHOUT hovering
# 2. Fatality rate number shown directly on map WITHOUT hovering
# 3. Full stats tooltip on hover (all 3 metrics)
# 4. Risk-colored borders per state (not just markers)
# 5. All 36 states labeled, not just top 10
# ============================================================

# All 36 state centroids (lat, lon)
STATE_CENTROIDS_ALL = {
    'Andaman & Nicobar Islands': (11.7401, 92.6586),
    'Andhra Pradesh':    (15.9129, 79.7400),
    'Arunachal Pradesh': (28.2180, 94.7278),
    'Assam':             (26.2006, 92.9376),
    'Bihar':             (25.0961, 85.3131),
    'Chandigarh':        (30.7333, 76.7794),
    'Chhattisgarh':      (21.2787, 81.8661),
    'D&N Haveli and Daman & Diu': (20.1809, 73.0169),
    'Delhi':             (28.7041, 77.1025),
    'Goa':               (15.2993, 74.1240),
    'Gujarat':           (22.2587, 71.1924),
    'Haryana':           (29.0588, 76.0856),
    'Himachal Pradesh':  (31.1048, 77.1734),
    'Jammu and Kashmir': (33.7782, 76.5762),
    'Jharkhand':         (23.6102, 85.2799),
    'Karnataka':         (15.3173, 75.7139),
    'Kerala':            (10.8505, 76.2711),
    'Ladakh':            (34.1526, 77.5770),
    'Lakshadweep':       (10.5667, 72.6417),
    'Madhya Pradesh':    (22.9734, 78.6569),
    'Maharashtra':       (19.7515, 75.7139),
    'Manipur':           (24.6637, 93.9063),
    'Meghalaya':         (25.4670, 91.3662),
    'Mizoram':           (23.1645, 92.9376),
    'Nagaland':          (26.1584, 94.5624),
    'Odisha':            (20.9517, 85.0985),
    'Puducherry':        (11.9416, 79.8083),
    'Punjab':            (31.1471, 75.3412),
    'Rajasthan':         (27.0238, 74.2179),
    'Sikkim':            (27.5330, 88.5122),
    'Tamil Nadu':        (11.1271, 78.6569),
    'Telangana':         (18.1124, 79.0193),
    'Tripura':           (23.9408, 91.9882),
    'Uttar Pradesh':     (26.8467, 80.9462),
    'Uttarakhand':       (30.0668, 79.0193),
    'West Bengal':       (22.9868, 87.8550),
}

# Add labels to EVERY state — name + fatality rate visible without hovering
for _, row in latest.iterrows():
    state = row['state']
    if state not in STATE_CENTROIDS_ALL:
        continue

    lat, lon = STATE_CENTROIDS_ALL[state]
    rate = row.get('fatality_rate_per_accident', np.nan)
    accidents = row.get('accidents', 0)
    fatalities = row.get('fatalities', 0)

    if pd.isna(rate):
        continue

    # Color coding by fatality rate
    if rate >= 0.7:
        dot_color = '#ff1744'   # bright red
        text_color = '#ff1744'
    elif rate >= 0.5:
        dot_color = '#ff6d00'   # orange
        text_color = '#ff6d00'
    elif rate >= 0.3:
        dot_color = '#ffd600'   # yellow
        text_color = '#e6c200'
    else:
        dot_color = '#00e676'   # green
        text_color = '#00c853'

    # DivIcon = custom HTML label pinned to map coordinates
    # Shows: State Name (top) + fatality rate (bottom)
    # Visible WITHOUT hovering
    label_html = f"""
        <div style="
            text-align: center;
            font-family: Arial, sans-serif;
            pointer-events: none;
        ">
            <div style="
                font-size: 9px;
                font-weight: bold;
                color: white;
                text-shadow: -1px -1px 0 #000, 1px -1px 0 #000,
                             -1px 1px 0 #000, 1px 1px 0 #000;
                line-height: 1.2;
                white-space: nowrap;
            ">{state.replace(' and ', ' & ')}</div>
            <div style="
                font-size: 10px;
                font-weight: bold;
                color: {text_color};
                text-shadow: -1px -1px 0 #000, 1px -1px 0 #000,
                             -1px 1px 0 #000, 1px 1px 0 #000;
            ">{rate:.3f}</div>
        </div>
    """

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=label_html,
            icon_size=(100, 35),
            icon_anchor=(50, 17),   # center the label on the coordinate
        ),
        # Rich hover tooltip with all 3 metrics
        tooltip=folium.Tooltip(
            f"""
            <div style='
                background:#1a1a2e; color:white;
                font-family:Arial; padding:8px 12px;
                border-radius:6px; border:1px solid {dot_color};
                font-size:13px; min-width:180px;
            '>
                <b style='font-size:14px; color:{dot_color}'>{state}</b><br>
                <hr style='border-color:#444; margin:4px 0'>
                🔴 Fatality Rate: <b>{rate:.4f}</b><br>
                🚗 Accidents 2024: <b>{int(accidents):,}</b><br>
                💀 Fatalities 2024: <b>{int(fatalities):,}</b>
            </div>
            """,
            sticky=True,
        )
    ).add_to(m1)

# Save updated map
m1.save(f'{OUTPUT}/india_risk_map.html')
print("✅ MAP SAVED with state labels + rates visible without hover")
print("   Open: brave-browser outputs/india_risk_map.html")

✅ MAP SAVED with state labels + rates visible without hover
   Open: brave-browser outputs/india_risk_map.html


In [11]:
m2 = folium.Map(
    location=[20.5937, 78.9629],
    zoom_start=5,
    tiles='CartoDB positron'
)

from folium import Choropleth
choropleth_data = latest[['state', 'fatality_rate_per_accident', 'accidents', 'fatalities']].copy()

Choropleth(
    geo_data=f'{OUTPUT}/india_mapped.geojson',
    data=choropleth_data,
    columns=['state', 'accidents'],
    key_on='feature.properties.state',
    fill_color='YlOrRd',
    fill_opacity=0.8,
    line_opacity=0.3,
    legend_name='Total Accidents 2024 (Raw Count)',
    nan_fill_color='lightgray',
    highlight=True,
).add_to(m2)

title_html2 = '''
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
     z-index:1000; background-color:white; padding: 10px 20px;
     border-radius:8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
     font-family: Arial; text-align:center;">
    <b style="font-size:16px">India Road Accidents — Raw Count 2024</b><br>
    <span style="font-size:12px; color:#666">Compare with Risk Map to see why normalization matters</span>
</div>
'''
m2.get_root().html.add_child(folium.Element(title_html2))

# Labels: state name + accident count visible without hovering
for _, row in latest.iterrows():
    state = row['state']
    if state not in STATE_CENTROIDS_ALL:
        continue
    lat, lon = STATE_CENTROIDS_ALL[state]
    accidents = row.get('accidents', 0)
    fatalities = row.get('fatalities', 0)
    rate = row.get('fatality_rate_per_accident', 0)
    if pd.isna(accidents):
        continue

    label_html = f"""
        <div style="text-align:center; font-family:Arial; pointer-events:none;">
            <div style="font-size:9px; font-weight:bold; color:white;
                text-shadow:-1px -1px 0 #000,1px -1px 0 #000,-1px 1px 0 #000,1px 1px 0 #000;
                line-height:1.2; white-space:nowrap;">
                {state.replace(' and ', ' & ')}
            </div>
            <div style="font-size:9px; font-weight:bold; color:#ffd700;
                text-shadow:-1px -1px 0 #000,1px -1px 0 #000,-1px 1px 0 #000,1px 1px 0 #000;">
                {int(accidents):,}
            </div>
        </div>
    """
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(html=label_html, icon_size=(100,30), icon_anchor=(50,15)),
        tooltip=folium.Tooltip(
            f"""<div style='background:#1a1a2e;color:white;font-family:Arial;
                padding:8px 12px;border-radius:6px;border:1px solid #ffd700;font-size:13px;'>
                <b style='font-size:14px;color:#ffd700'>{state}</b><br>
                <hr style='border-color:#444;margin:4px 0'>
                🚗 Accidents 2024: <b>{int(accidents):,}</b><br>
                💀 Fatalities 2024: <b>{int(fatalities):,}</b><br>
                🔴 Fatality Rate: <b>{rate:.4f}</b>
            </div>""",
            sticky=True
        )
    ).add_to(m2)

m2.save(f'{OUTPUT}/india_accidents_map.html')
print("✅ MAP 2 SAVED with state name + accident count labels without hover")
print("   Open: brave-browser outputs/india_accidents_map.html")


✅ MAP 2 SAVED with state name + accident count labels without hover
   Open: brave-browser outputs/india_accidents_map.html
